# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane (Refresh / Content Opportunity Scoring) is fundamentally a **ranking/scoring problem built 
on top of a classification model**.

- The core task type is **classification**: for each page, predict the probability that it belongs 
  to a "needs review" class (e.g. `is_declining_label`).
- That probability is then used to **rank** all pages, because a reviewer with limited time doesn't 
  need a yes/no answer for every page — they need an ordered queue: "look at this one first, then 
  this one."
- So the output isn't just a label — it's a **score** used to sort a review queue, with reason codes 
  attached so a human can see why a page was flagged.

This matches the starter pipeline exactly: it trains a classifier (`is_declining_label`), then turns 
its output probability into `final_refresh_score`, which produces the ranked queue.

## 2. Target or proxy

**Starter proxy target:** `is_declining_label = (trend_direction == "down")`

This is a **proxy**, not a true future outcome — it's calculated from the *current* 90-day window, 
not from what happens *after* someone decides to act. It tells us "this page currently looks like 
it's declining," not "this page will keep declining if nobody touches it."

**Why it's useful anyway (for now):** it gives a working, checkable label to build a first model 
against, and it correlates with things a reviewer would actually care about (54.2% of the starter 
slice carries this label, so it's a meaningful, non-trivial split).

**Where I want to take it:** a stronger version of this target would use a future window, e.g.:

## 3. Success metric

**Primary metric: Precision@50** (and Precision@20 as a secondary check).

**Why this metric, not plain accuracy:** a reviewer doesn't look at every page — they can only act 
on a fixed number of top candidates (e.g. the top 50 the queue surfaces). Precision@K directly 
answers "of the top K pages I'm about to spend my limited time reviewing, how many were actually 
worth reviewing?" — which matches how the output is actually used.

Plain accuracy would be misleading here: with 54.2% of pages already "declining," a model could get 
decent accuracy just by guessing the majority class, without producing a genuinely useful ranking.

**Reference point from the starter pipeline:**
- Baseline rule: Precision@50 = 0.240 (about 12 of the top 50 are true positives)
- Random forest: Precision@50 = 0.740 (about 37 of the top 50 are true positives)

I'll also track **recall** as a secondary check, since missing a genuinely declining page (a false 
negative) means lost traffic keeps compounding silently — so I don't want to only optimize for a 
tight, "safe" top-K at the cost of missing real problems further down the list.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# One row = one content page, described by its 90-day window of signals
unit_of_analysis = df[[
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "content_age_days", "trend_direction", "avg_position", "ctr"
]].head(10)

unit_of_analysis

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,avg_position,ctr
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,10.6,0.76
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,20.3,0.05
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,36.5,0.09
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,6.2,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,44.0,0.13
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,8.5,0.03
6,content_9a34b442b552,client_8722616204,20,1,90,down,7.0,0.00
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,21.2,0.06
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,46.0,0.09
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,4.9,0.16


## 4. The unit of analysis, as a real dataframe

One row = **one content page**, summarized by its 90-day window of observed signals. The table above 
shows 10 real rows from the starter dataset — for example, row 0 (`content_304f48230142`) has 3,803 
impressions, 17 sessions, is 187 days old, currently trending "down," sits at average position 10.6, 
with a CTR of 0.76. This is exactly the grain my model scores and ranks: a page, not a client, not a 
query, not a day.

Notice the variety already visible in just these 10 rows — row 6 (`content_9a34b442b552`) has only 
20 impressions total, while row 8 (`content_5e6c160719bc`) has 32,574 — a huge range in scale, which 
is why volume filters (like `impressions_90d >= 500` in the baseline rules) matter before treating 
every "down" trend the same way.

## 5. Why ML beats a fixed rule here

A fixed rule (like the starter baseline) can only combine a handful of signals in a way a human 
wrote down in advance — e.g. "flag if impressions >= 500 AND days since update >= 180." It can't 
discover that, say, position 6-10 combined with low CTR *and* recent word-count growth behaves very 
differently from position 6-10 alone.

The starter pipeline's own numbers make this concrete: the hand-written baseline rule reaches 
Precision@50 = 0.240, while a random forest trained on the same signals reaches 0.740 — roughly 3x 
more true positives in the same top-50 slots. That gap is real evidence that useful patterns exist 
in the *combinations* of signals that a fixed rule doesn't capture, and that a model earns its 
complexity here rather than just adding it for its own sake.

This is not "just run a rule," because the whole reason to build a model is that the rule alone 
already left most of the value on the table (0.240 vs 0.740 is not a marginal difference).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.